# Energy Transition & Electric Mobility in Portugal - Cleaning

**Phase 2: Cleaning**

Transforms the raw REN JSON (Highcharts format: one time series per energy
source, 96 fifteen-minute readings per day) into a tidy daily-aggregate
table — one row per date + energy source.

In [1]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)

## Load the raw REN data

Each entry is `{"date": ..., "data": {...}}`, where `data` holds the
Highcharts structure: `xAxis.categories` (96 time labels shared by every
series) and `series` (a list of energy sources, each with 96 MW readings).

In [2]:
with open(DATA_RAW / "ren_production_breakdown_raw.json") as f:
    ren_raw = json.load(f)

print(f"Loaded {len(ren_raw)} days")

Loaded 1097 days


## Convert one day into tidy rows

Power (MW) is converted to energy (MWh) by multiplying by 0.25 - each
reading represents a 15-minute interval, i.e. 0.25 hours.

This function treats every series the same way (Hydro, Solar, Consumption,
Import, Export, batteries...) without judging which ones count as
"renewable production" - that classification belongs to the analysis step
later, not to cleaning.

In [3]:
def process_ren_day(day_entry):
    """
    Convert one day of REN data into daily-aggregate rows — one row per
    energy source, with total energy, average/peak power, and peak time.
    """
    day_date = day_entry["date"]
    day_data = day_entry["data"]

    # The 96 time labels (00:00, 00:15, ..., 23:45) are shared by every series
    time_labels = day_data["xAxis"]["categories"]

    rows = []

    for series in day_data["series"]:
        source_name = series["name"]
        values = series["data"]  # list of 96 MW readings, may contain None

        # Drop missing readings before doing math on them — a day with a
        # few gaps shouldn't crash the whole pipeline
        valid_pairs = [
            (time_labels[i], v) for i, v in enumerate(values) if v is not None
        ]

        if not valid_pairs:
            # entire series missing for this day — skip it, don't fabricate a zero
            continue

        valid_times = [t for t, v in valid_pairs]
        valid_values = [v for t, v in valid_pairs]

        # Power (MW) -> Energy (MWh): each reading = 15 min = 0.25 h
        energia_mwh = sum(valid_values) * 0.25
        potencia_media_mw = sum(valid_values) / len(valid_values)
        potencia_pico_mw = max(valid_values)

        # time label matching the peak value (first occurrence if tied)
        peak_index = valid_values.index(potencia_pico_mw)
        hora_pico = valid_times[peak_index]

        rows.append(
            {
                "date": day_date,
                "source": source_name,
                "energia_mwh": round(energia_mwh, 2),
                "potencia_media_mw": round(potencia_media_mw, 2),
                "potencia_pico_mw": potencia_pico_mw,
                "hora_pico": hora_pico,
            }
        )

    return rows

## Run it over every day

In [4]:
all_rows = []
for day_entry in ren_raw:
    all_rows.extend(process_ren_day(day_entry))

df_ren_daily = pd.DataFrame(all_rows)

print(f"Total rows: {len(df_ren_daily)}")  # ~1097 days x up to 15 sources
print(f"Distinct sources: {df_ren_daily['source'].nunique()}")
df_ren_daily.head(10)

Total rows: 16214
Distinct sources: 15


,date,source,energia_mwh,potencia_media_mw,potencia_pico_mw,hora_pico
0,2023-09-16,Consumption + Storage,130599.50,5441.65,7036.9,13:15
1,2023-09-16,Consumption,122490.15,5103.76,6093.1,20:15
2,2023-09-16,Pumping,8109.35,337.89,1800.4,14:15
3,2023-09-16,Hydro,22007.58,916.98,3627.2,19:45
4,2023-09-16,Solar,5671.68,236.32,886.9,13:45
5,2023-09-16,Import,21219.53,884.15,2386.2,14:00
6,2023-09-16,Export,1510.80,62.95,775.3,19:00
7,2023-09-16,Wind,40404.28,1683.51,2490.9,10:00
8,2023-09-16,Natural Gas,34906.55,1454.44,2227.7,04:15
9,2023-09-16,Other Thermal,311.68,12.99,13.4,18:30


## Classify sources and calculate daily renewable %

Pivot the long table (one row per source) into one row per date, then
compute the renewable share using only production sources - demand/flow
series (Consumption, Import, Export, Pumping, batteries) are excluded.

In [ ]:
RENEWABLE_SOURCES = ["Hydro", "Solar", "Wind", "Biomass", "Wave"]
NON_RENEWABLE_SOURCES = ["Natural Gas", "Other Thermal", "Coal"]
# Everything else (Consumption, Import, Export, Pumping, battery flows) is
# demand/flow, not production - excluded from the renewable % calculation

# Pivot: one row per date, one column per source, values = energia_mwh
df_wide = df_ren_daily.pivot(index="date", columns="source", values="energia_mwh")

df_wide["renovavel_mwh"] = df_wide[RENEWABLE_SOURCES].sum(axis=1)
df_wide["nao_renovavel_mwh"] = df_wide[NON_RENEWABLE_SOURCES].sum(axis=1)
df_wide["producao_total_mwh"] = df_wide["renovavel_mwh"] + df_wide["nao_renovavel_mwh"]
df_wide["pct_renovavel"] = (
    df_wide["renovavel_mwh"] / df_wide["producao_total_mwh"] * 100
).round(1)

print(f"Date range: {df_wide.index.min()} to {df_wide.index.max()}")
print(f"Average renewable %: {df_wide['pct_renovavel'].mean():.1f}%")
df_wide[["renovavel_mwh", "nao_renovavel_mwh", "pct_renovavel"]].head(10)

Date range: 2023-09-16 to 2026-09-16
Average renewable %: 85.0%


source,renovavel_mwh,nao_renovavel_mwh,pct_renovavel
date,,,
2023-09-16,75957.11,35218.23,68.3
2023-09-17,107501.32,7047.02,93.8
2023-09-18,55272.42,48771.60,53.1
2023-09-19,42004.95,50146.22,45.6
2023-09-20,68035.76,37005.64,64.8
2023-09-21,104330.66,18639.48,84.8
2023-09-22,70634.24,23200.15,75.3
2023-09-23,59102.18,19375.55,75.3
2023-09-24,57356.30,12918.25,81.6
